In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
import community as community_louvain
import os

DATA_DIR = "./data"
OUTPUT_DIR = "./outputs"
 
pd.set_option('display.max_columns', None)


In [ ]:
# Load the weighted edge list
edges = pd.read_excel(os.path.join(OUTPUT_DIR, 'weighted_edges.xlsx'))
print(f"Total edge rows: {len(edges)}")
print(f"Columns: {edges.columns.tolist()}")

Total edge rows: 1887
Columns: ['source', 'target', 'sentiment', 'weight']


In [3]:
# Identify self-loops
self_loops = edges[edges['source'] == edges['target']]
print(f"Self-loop rows: {len(self_loops)}")
print(f"Self-loop unique source-target pairs: {self_loops[['source','target']].drop_duplicates().shape[0]}")

Self-loop rows: 506
Self-loop unique source-target pairs: 402


In [4]:
# Identify statistical outlier accounts by raw out-degree (row count)
out_degree_raw = edges.groupby('source').size()
mean_od = out_degree_raw.mean()
std_od = out_degree_raw.std()
threshold = mean_od + 3 * std_od

In [5]:
print(f"Mean out-degree: {mean_od:.2f}")
print(f"Std out-degree: {std_od:.2f}")
print(f"3-sigma threshold: {threshold:.2f}")

Mean out-degree: 1.71
Std out-degree: 3.03
3-sigma threshold: 10.79


In [6]:
outlier_accounts = out_degree_raw[out_degree_raw > threshold].sort_values(ascending=False)
print(f"\nAccounts exceeding 3-sigma threshold ({len(outlier_accounts)} accounts):")
print(outlier_accounts)


Accounts exceeding 3-sigma threshold (5 accounts):
source
grok              90
rielbrazoes       22
nanangshmhmkn     19
tamerlane9470     12
abanggeutanyoe    11
dtype: int64


In [7]:
outlier_account_list = outlier_accounts.index.tolist()

In [8]:
def build_graph(edge_df):
    """Build a directed weighted graph from an edge dataframe."""
    return nx.from_pandas_edgelist(
        edge_df, source='source', target='target',
        edge_attr='weight', create_using=nx.DiGraph()
    )

In [9]:
def compute_metrics(G, label, top_n=10):
    """Compute structural metrics, community detection, and centrality 
    rankings for a given directed graph."""
    n = G.number_of_nodes()
    e = G.number_of_edges()
    avg_deg = 2 * e / n
    density = nx.density(G)
 
    G_undirected = G.to_undirected()
    if nx.is_connected(G_undirected):
        apl = nx.average_shortest_path_length(G_undirected)
        diameter = nx.diameter(G_undirected)
        connected_note = "fully connected"
    else:
        largest_cc = max(nx.connected_components(G_undirected), key=len)
        sub = G_undirected.subgraph(largest_cc)
        apl = nx.average_shortest_path_length(sub)
        diameter = nx.diameter(sub)
        connected_note = f"largest component only ({len(largest_cc)}/{n} nodes)"
 
    partition = community_louvain.best_partition(G_undirected, random_state=42)
    n_communities = len(set(partition.values()))
    modularity = community_louvain.modularity(partition, G_undirected)
 
    pagerank = nx.pagerank(G)
    top_pagerank = sorted(pagerank.items(), key=lambda x: -x[1])[:top_n]
 
    in_degree = dict(G.in_degree())
    top_in_degree = sorted(in_degree.items(), key=lambda x: -x[1])[:top_n]
 
    out_degree = dict(G.out_degree())
    top_out_degree = sorted(out_degree.items(), key=lambda x: -x[1])[:top_n]
 
    betweenness = nx.betweenness_centrality(G)
    top_betweenness = sorted(betweenness.items(), key=lambda x: -x[1])[:top_n]
 
    results = {
        'label': label,
        'nodes': n, 'edges': e,
        'avg_degree': round(avg_deg, 4),
        'density': round(density, 6),
        'avg_path_length': round(apl, 4),
        'diameter': diameter,
        'connectivity_note': connected_note,
        'communities': n_communities,
        'modularity': round(modularity, 4),
        'top_pagerank': top_pagerank,
        'top_in_degree': top_in_degree,
        'top_out_degree': top_out_degree,
        'top_betweenness': top_betweenness,
    }
 
    print(f"\n{'='*70}")
    print(f"  {label}")
    print(f"{'='*70}")
    print(f"Nodes: {n} | Edges: {e}")
    print(f"Avg Degree: {avg_deg:.4f} | Density: {density:.6f}")
    print(f"Avg Path Length: {apl:.4f} ({connected_note}) | Diameter: {diameter}")
    print(f"Communities: {n_communities} | Modularity: {modularity:.4f}")
    print(f"\nTop {top_n} PageRank:")
    for u, s in top_pagerank:
        print(f"  {u}: {s:.6f}")
    print(f"\nTop {top_n} In-Degree:")
    for u, s in top_in_degree:
        print(f"  {u}: {s}")
    print(f"\nTop {top_n} Out-Degree:")
    for u, s in top_out_degree:
        print(f"  {u}: {s}")
    print(f"\nTop {top_n} Betweenness:")
    for u, s in top_betweenness:
        print(f"  {u}: {s:.8f}")
 
    return results

In [10]:
def get_sentiment_edges(edge_df, sentiment=None):
    """Return edges for a given sentiment, or all edges if sentiment=None
    (i.e., the complete network)."""
    if sentiment is None:
        return edge_df
    return edge_df[edge_df['sentiment'] == sentiment]
 
networks_to_test = {
    'Complete': None,
    'Positive': 'positive',
    'Neutral': 'neutral',
    'Negative': 'negative',
}

In [15]:
all_results = {}

In [16]:
for network_name, sentiment_filter in networks_to_test.items():
    print(f"\n\n{'#'*70}")
    print(f"#  NETWORK: {network_name}")
    print(f"{'#'*70}")
 
    base_edges = get_sentiment_edges(edges, sentiment_filter)
 
    # Recompute self-loops and outlier accounts WITHIN this network's edges
    # (an account might be a high-volume outlier in one sentiment network
    # but not another, so thresholds are recalculated per network)
    self_loops_net = base_edges[base_edges['source'] == base_edges['target']]
    out_degree_net = base_edges.groupby('source').size()
    mean_net = out_degree_net.mean()
    std_net = out_degree_net.std()
    threshold_net = mean_net + 3 * std_net
    outliers_net = out_degree_net[out_degree_net > threshold_net].index.tolist()
 
    print(f"\nSelf-loop rows in this network: {len(self_loops_net)}")
    print(f"Outlier accounts in this network ({len(outliers_net)}): {outliers_net}")
 
    # Condition 1: Original
    G_orig = build_graph(base_edges)
    res1 = compute_metrics(G_orig, f"{network_name} - CONDITION 1: ORIGINAL")
 
    # Condition 2: Without self-loops
    edges_no_loops_net = base_edges[base_edges['source'] != base_edges['target']]
    G_noloop = build_graph(edges_no_loops_net)
    res2 = compute_metrics(G_noloop, f"{network_name} - CONDITION 2: NO SELF-LOOPS")
 
    # Condition 3: Without self-loops + without outlier accounts
    edges_filt_net = edges_no_loops_net[
        (~edges_no_loops_net['source'].isin(outliers_net)) &
        (~edges_no_loops_net['target'].isin(outliers_net))
    ]
    G_filt = build_graph(edges_filt_net)
    res3 = compute_metrics(G_filt, f"{network_name} - CONDITION 3: NO SELF-LOOPS + NO OUTLIERS")
 
    all_results[network_name] = {
        'original': res1,
        'no_selfloops': res2,
        'filtered': res3,
        'outlier_accounts': outliers_net,
        'self_loop_count': len(self_loops_net),
    }



######################################################################
#  NETWORK: Complete
######################################################################

Self-loop rows in this network: 506
Outlier accounts in this network (5): ['abanggeutanyoe', 'grok', 'nanangshmhmkn', 'rielbrazoes', 'tamerlane9470']

  Complete - CONDITION 1: ORIGINAL
Nodes: 2166 | Edges: 1767
Avg Degree: 1.6316 | Density: 0.000377
Avg Path Length: 5.4838 (largest component only (172/2166 nodes)) | Diameter: 12
Communities: 861 | Modularity: 0.9868

Top 10 PageRank:
  afrkml: 0.003688
  tirta_cipeng: 0.003668
  dinkesjateng: 0.003091
  kegblgnunfaedh: 0.002749
  convomf: 0.002663
  merdekadotcom: 0.002514
  tanyarlfes: 0.002281
  tanyakanrl: 0.002197
  angewwie: 0.002129
  bertanyarl: 0.001843

Top 10 In-Degree:
  KemenkesRI: 19
  kegblgnunfaedh: 16
  convomf: 16
  tanyakanrl: 13
  tanyarlfes: 13
  bertanyarl: 10
  jokowi: 6
  gayung_aer: 6
  amti_d19: 6
  thinkway_ID: 6

Top 10 Out-Degree:
  grok: 84
  

In [17]:
summary_rows = []
for network_name, conditions in all_results.items():
    for cond_key, r in [('original', conditions['original']),
                          ('no_selfloops', conditions['no_selfloops']),
                          ('filtered', conditions['filtered'])]:
        summary_rows.append({
            'Network': network_name,
            'Condition': cond_key,
            'Nodes': r['nodes'],
            'Edges': r['edges'],
            'Avg Degree': r['avg_degree'],
            'Density': r['density'],
            'Avg Path Length': r['avg_path_length'],
            'Diameter': r['diameter'],
            'Communities': r['communities'],
            'Modularity': r['modularity'],
            'Top PageRank User': r['top_pagerank'][0][0],
            'Top PageRank Score': round(r['top_pagerank'][0][1], 6),
            'Top In-Degree User': r['top_in_degree'][0][0],
            'Top In-Degree Score': r['top_in_degree'][0][1],
            'Top Betweenness User': r['top_betweenness'][0][0],
            'Top Betweenness Score': round(r['top_betweenness'][0][1], 8),
        })

In [18]:
summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

 Network    Condition  Nodes  Edges  Avg Degree  Density  Avg Path Length  Diameter  Communities  Modularity Top PageRank User  Top PageRank Score Top In-Degree User  Top In-Degree Score Top Betweenness User  Top Betweenness Score
Complete     original   2166   1767      1.6316 0.000377           5.4838        12          861      0.9868            afrkml            0.003688         KemenkesRI                   19        drevachaniago               0.000002
Complete no_selfloops   1821   1365      1.4992 0.000412           5.4838        12          516      0.9821    kegblgnunfaedh            0.005389         KemenkesRI                   19        drevachaniago               0.000003
Complete     filtered   1687   1225      1.4523 0.000431           5.2136        11          516      0.9892    kegblgnunfaedh            0.005721         KemenkesRI                   18        drevachaniago               0.000004
Positive     original    378    318      1.6825 0.002231           3.7525   

In [ ]:
summary_df.to_excel(os.path.join(OUTPUT_DIR, 'sensitivity_analysis_summary_all_networks.xlsx'), index=False)

In [ ]:
## Step 5: Save Full Top-10 Rankings for Each Network x Condition
def rankings_to_df(result, metric_key):
    return pd.DataFrame(result[metric_key], columns=['username', 'score'])
 
with pd.ExcelWriter(os.path.join(OUTPUT_DIR, 'sensitivity_analysis_full_rankings_all_networks.xlsx')) as writer:
    for network_name, conditions in all_results.items():
        for cond_label, r in [('original', conditions['original']),
                                ('no_selfloops', conditions['no_selfloops']),
                                ('filtered', conditions['filtered'])]:
            prefix = f"{network_name[:4]}_{cond_label[:10]}"
            rankings_to_df(r, 'top_pagerank').to_excel(
                writer, sheet_name=f'{prefix}_pr'[:31], index=False)
            rankings_to_df(r, 'top_in_degree').to_excel(
                writer, sheet_name=f'{prefix}_ind'[:31], index=False)
            rankings_to_df(r, 'top_betweenness').to_excel(
                writer, sheet_name=f'{prefix}_bw'[:31], index=False)
 
print("\nFiles saved:")
print("1. sensitivity_analysis_summary_all_networks.xlsx")
print("2. sensitivity_analysis_full_rankings_all_networks.xlsx")


Files saved:
1. sensitivity_analysis_summary_all_networks.xlsx
2. sensitivity_analysis_full_rankings_all_networks.xlsx


In [21]:
def diagnose_account(username, edge_df):
    """Print all edges involving a given account, split by self-loop vs not."""
    incoming = edge_df[edge_df['target'] == username]
    outgoing = edge_df[edge_df['source'] == username]
    self_loop = edge_df[(edge_df['source'] == username) & (edge_df['target'] == username)]
 
    print(f"\n--- Diagnostic for @{username} ---")
    print(f"Total incoming edges: {len(incoming)}")
    print(f"Total outgoing edges: {len(outgoing)}")
    print(f"Self-loop edges: {len(self_loop)}")
    print(f"\nIncoming edges detail:")
    print(incoming[['source', 'target', 'sentiment', 'weight']])

In [22]:
diagnose_account('afrkml', edges)


--- Diagnostic for @afrkml ---
Total incoming edges: 5
Total outgoing edges: 2
Self-loop edges: 2

Incoming edges detail:
         source  target sentiment  weight
66       afrkml  afrkml  negative       1
67       afrkml  afrkml   neutral       1
507    erninayy  afrkml  negative       2
623        grok  afrkml  negative       2
1387  pooriam__  afrkml  positive       2
